# Parking Spaces in Berlin 

# Step 2: Data transformation & Preprocessing

**Goals:**

- Begin the main transformation and preprocessing of approved Parking Spaces data sources.

- Use OSM data via OSMnx as the primary dataset, comparing and enriching it with other sources where possible.

- Integrate multiple datasets into a unified, consistent format.

- Determine the final database schema (tables structure) for the unified parking dataset.

- Prepare the cleaned and preprocessed data for loading into the database.


**Prepare notebook environment:**

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import unary_union
import os
import datetime as dt
from datetime import datetime, timezone
import osmnx as ox  
import re
from typing import Union

# Always work in WGS84
TARGET_CRS = "EPSG:4326"
RUN_TS = datetime.now(timezone.utc).isoformat()

pd.set_option("display.max_columns", 100)

## 2.1 Data Export & Cleaning

We will load the OSM parking data (main dataset) and the enrichment datasets from Berlin Open Data.

Before merging datasets from OSM and Berlin Open Data, all column names are standardized to ensure consistency across sources.

### 2.1.1 Data Loading

#### Load OSM data (main dataset)

In [2]:
TagValue = Union[bool, str, list[str]]

# Off-street style parking: amenity=parkin
tags_parking: dict[str, TagValue] = {"amenity": "parking"}
parking_gdf = ox.features_from_place("Berlin, Germany", tags_parking)
print(f"Off-street parking objects fetched: {len(parking_gdf)}")

# Individual/on-street parking spaces: amenity=parking_space
tags_parking_space: dict[str, TagValue] = {"amenity": "parking_space"}
parking_space_gdf = ox.features_from_place("Berlin, Germany", tags_parking_space)
print(f"Parking space objects fetched: {len(parking_space_gdf)}")

# Parking entrances (garages, underground, etc.)
tags_parking_entrance: dict[str, TagValue] = {"amenity": "parking_entrance"}
parking_entrance_gdf = ox.features_from_place("Berlin, Germany", tags_parking_entrance)
print(f"Parking entrance objects fetched: {len(parking_entrance_gdf)}")

Off-street parking objects fetched: 46136
Parking space objects fetched: 5294
Parking entrance objects fetched: 2478


#### Load Berlin Open Data (enrichment datasets)

**Helper function to load Berlin Open Data:**

In [3]:
TARGET_CRS = "EPSG:4326"
SOURCES_DIR = Path("../sources")

def load_or_fetch_wfs(local_name: str, wfs_url: str) -> gpd.GeoDataFrame:
    """
    Load a GeoJSON from /sources if present, otherwise fetch from WFS,
    save it, and return as GeoDataFrame in TARGET_CRS.
    """
    local_path = SOURCES_DIR / local_name

    if local_path.exists():
        gdf = gpd.read_file(local_path)
        print(f"➡️ loaded local {local_name} → {len(gdf)} rows")
    else:
        print(f"🌐 fetching from WFS → {wfs_url}")
        gdf = gpd.read_file(wfs_url)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        gdf.to_file(local_path, driver="GeoJSON")
        print(f"✅ saved to {local_path}")

    return gdf.to_crs(TARGET_CRS)

**Helper function to load street parking data from Berlin Open Data:**

*Berlin Open Data Street Parking has seperate data from inside and outside the Berlin S-Bahn and needs to be loaded differently than the other Berlin Open Datasets.*

In [4]:
def load_berlin_street_parking():
    """
    Load Berlin street parking (inside + outside S-Bahn ring).
    If local cached file exists, use it.
    Otherwise fetch the two WFS layers separately and merge.
    """
    local_path = SOURCES_DIR / "bod_parking_street.geojson"
    if local_path.exists():
        print(f"📁 loading local street parking → {local_path}")
        gdf = gpd.read_file(local_path).to_crs(TARGET_CRS)
        return gdf

    base = "https://gdi.berlin.de/services/wfs/parkplaetze"
    common = "?service=WFS&version=2.0.0&request=GetFeature&outputFormat=application/json"

    inside_url = base + common + "&typeNames=parkplaetze:parkplaetze"
    outside_url = base + common + "&typeNames=parkplaetze:parkplaetze_aussen"

    print("🌐 fetching inside ring…")
    gdf_inside = gpd.read_file(inside_url)
    print("🌐 fetching outside ring…")
    gdf_outside = gpd.read_file(outside_url)

    parking_all = gpd.GeoDataFrame(
        pd.concat([gdf_inside, gdf_outside], ignore_index=True),
        crs=gdf_inside.crs,
    ).to_crs(TARGET_CRS)

    # cache it
    local_path.parent.mkdir(parents=True, exist_ok=True)
    parking_all.to_file(local_path, driver="GeoJSON")
    print(f"💾 cached merged street parking → {local_path}")

    return parking_all

In [5]:
# Berlin street parking (inside + outside Ring Bahn)
bod_parking_street = load_berlin_street_parking()

# Park & Ride
bod_park_and_ride = load_or_fetch_wfs(
    "bod_park_and_ride.geojson",
    (
        "https://gdi.berlin.de/services/wfs/park_and_ride"
        "?service=WFS&version=2.0.0&request=GetFeature"
        "&typenames=park_and_ride:park_and_ride"
        "&outputFormat=application/json"
    ),
)

# Managed parking zones
bod_parking_zones = load_or_fetch_wfs(
    "bod_parking_zones.geojson",
    (
        "https://gdi.berlin.de/services/wfs/parkraumbewirtschaftung"
        "?service=WFS&version=2.0.0&request=GetFeature"
        "&typenames=parkraumbewirtschaftung:parkzonen"
        "&outputFormat=application/json"
    ),
)

loaded = {
    "bod_parking_street": bod_parking_street,
    "bod_park_and_ride": bod_park_and_ride,
    "bod_parking_zones": bod_parking_zones,
}
print("✅ Loaded sources:", loaded.keys())

🌐 fetching inside ring…
🌐 fetching outside ring…
💾 cached merged street parking → ../sources/bod_parking_street.geojson
🌐 fetching from WFS → https://gdi.berlin.de/services/wfs/park_and_ride?service=WFS&version=2.0.0&request=GetFeature&typenames=park_and_ride:park_and_ride&outputFormat=application/json
✅ saved to ../sources/bod_park_and_ride.geojson
🌐 fetching from WFS → https://gdi.berlin.de/services/wfs/parkraumbewirtschaftung?service=WFS&version=2.0.0&request=GetFeature&typenames=parkraumbewirtschaftung:parkzonen&outputFormat=application/json
✅ saved to ../sources/bod_parking_zones.geojson
✅ Loaded sources: dict_keys(['bod_parking_street', 'bod_park_and_ride', 'bod_parking_zones'])


### 2.1.2 Standardize column names

This step:
- Removes whitespace and special characters  

- Converts CamelCase or PascalCase to `snake_case`  

- Converts all names to lowercase  

➡️ *This guarantees that subsequent merge and join operations can run without case or formatting mismatches.*

**Helper function to clean column names:**

In [6]:
# Function to standardize column names 
def clean_column_names(name: str) -> str:
    name = name.strip()                                   # remove leading and trailing white space
    name = re.sub(r"[^\w]+", "_", name)                   # replace special characters
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)   # handle camelcase and pascalcase 
    return name.lower().strip("_")                        # lowercase and remove leading and trailing underscores

**Clean column names for loaded datasetes:**

In [7]:
# standardize OSM columns
parking_gdf.columns = [clean_column_names(c) for c in parking_gdf.columns]
parking_space_gdf.columns = [clean_column_names(c) for c in parking_space_gdf.columns]
parking_entrance_gdf.columns = [clean_column_names(c) for c in parking_entrance_gdf.columns]

# standardize columns Berlin Open Data
for key, gdf in loaded.items():
    gdf.columns = [clean_column_names(c) for c in gdf.columns]
    loaded[key] = gdf

## 2.2 Data Transformation & Integration

In this step, we enrich and integrate all available parking-related sources to build a unified, geospatially consistent dataset for Berlin.

The workflow combines data from:

- **OpenStreetMap (OSM)** – base layer for parking locations  

- **Berlin Open Data:**

  - Parking Zones  
  - Park & Ride Areas  
  - Street Parking  

The objective is to:

1. Enrich the OSM dataset with attributes from Berlin Open Data sources  
2. Remove duplicates (attribute and spatial)  
3. Verify the completeness and spatial alignment of districts and subdistricts  
4. Ensure final data consistency before schema design  

### Overview of Subsections

| Step | Description |
|------|--------------|
| **2.2.1 Enrichment & Source Comparison** | Combine OSM and Berlin Open Data layers, assess overlaps and coverage |
| **2.2.2 Create Unified Dataset** | Merge sources, deduplicate features, and align key attributes |
| **2.2.3 Verify Data Consistency** | Validate geometry, nulls, CRS, and create verification flags |

### 2.2.1 Enrichment & Source Comparison

**Combine the thee OSM layers into single GeoDataFrame and make sure they have the same CRS:**

In [8]:
# Ensure all are in WGS84
for gdf in [parking_gdf, parking_space_gdf, parking_entrance_gdf]:
    gdf.to_crs(TARGET_CRS, inplace=True)

# Add a parking_type column to keep them distinguishable
parking_gdf["parking_type"] = "off_street"
parking_space_gdf["parking_type"] = "on_street"
parking_entrance_gdf["parking_type"] = "entrance"

# Combine into one OSM GeoDataFrame
osm_all = pd.concat([parking_gdf, parking_space_gdf, parking_entrance_gdf], ignore_index=True)
osm_all = gpd.GeoDataFrame(osm_all, geometry="geometry", crs=TARGET_CRS)
print(f"✅ Combined OSM dataset: {len(osm_all)} records")

✅ Combined OSM dataset: 53908 records


➡️ Now osm_all contains all OSM parking-related geometries and will be the primary dataset for enrichment.


#### Berlin Parking Zones Enrichment

**Load and inspect the Berlin parking zones:**

In [9]:
zones_gdf = loaded["bod_parking_zones"]
print(f"✅ Parking zones loaded: {len(zones_gdf)} polygons")
zones_gdf.head()

✅ Parking zones loaded: 94 polygons


,id,parkzone,bezirk,zeiten,gebuehr,bemerkung,geometry
0,parkzonen.1,1,Mitte,Mo-Sa 9-22 Uhr,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((13.39148 52.52266, 13.39148 52..."
1,parkzonen.10,10,Spandau,"Mo-Fr 9-17 Uhr, Sa 9 -14 Uhr/ Advents-Sa 9 -17...","2,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((13.19977 52.53441, 13.20081 52..."
2,parkzonen.100,100,Neukölln,Mo-Fr 9-20 Uhr\n,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((13.44013 52.4814, 13.44034 52...."
3,parkzonen.101,101,Neukölln,Mo-Fr 9-20 Uhr,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((13.43552 52.48069, 13.43376 52..."
4,parkzonen.105,105,Neukölln,Mo-Fr 9-20 Uhr\n,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...,"MULTIPOLYGON (((13.42551 52.48812, 13.42513 52..."


**Compare geometry type:**

In [10]:
print(zones_gdf.crs)
print(osm_all.crs)

EPSG:4326
EPSG:4326


**Perform spatial join to attach zone info to every OSM parking geometry that lies within a Berlin parking zone polygon:**

➡️ This will:

- Keep all rows from osm_all (how="left")

- Add columns from zones_gdf where the OSM geometry is within a zone polygon

- Result in a combined GeoDataFrame where every OSM parking now has the zone’s attributes attached (e.g. zone ID, fee, time restriction, etc.)

In [11]:
osm_all = gpd.GeoDataFrame(osm_all, geometry="geometry", crs="EPSG:4326")
zones_gdf = gpd.GeoDataFrame(zones_gdf, geometry="geometry", crs="EPSG:4326")

osm_with_zones = gpd.sjoin(
    osm_all,
    zones_gdf,
    how="left",
    predicate="within",
).copy()

/Users/didodeboodt/GitHub Reposotories/layered-populate-data-pool-da/.venv/lib/python3.13/site-packages/geopandas/tools/sjoin.py:266: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_reset.reset_index(inplace=True)


In [12]:
osm_with_zones.head()

,geometry,access,addr_city,addr_country,addr_housenumber,addr_postcode,addr_street,addr_suburb,amenity,fee,maxheight,name,opening_hours,parking,website,wheelchair,note,barrier,capacity,wheelchair_description,toilets_wheelchair,operator,operator_type,operator_wikidata,created_by,capacity_disabled,capacity_parent,capacity_women,url,level,check_date_fee,check_date,foot,motorcycle,trailer,layer,lit,alt_name,maxstay,description,phone,source,check_date_opening_hours,park_ride,covered,maxweight,parking_condition,brand,brand_wikidata,brand_wikipedia,...,symbol,parking_description,fid,parking_left_access,parking_left_fee,lane_markings,library_bus_conditional,restriction_taxi,authentication_disc,mobile_library_conditional,restriction_bus,capacity_taxi,private_conditional,width_lane,width_street_side,cycleway_both,maxspeed_type,name_etymology_wikidata,sidewalk_left,source_maxspeed,zone_maxspeed,capacity_emergency,parking_both_informal,noname,parking_type,access_disabled,parking_disabled,entrance,maxlength,maxheight_signed,source_addr,horse,gate_type,bicycle_parking,door,mofa,moped,vehicle,exit,caravans,incline,source_maxheight,maxwidth,index_right,id,parkzone,bezirk,zeiten,gebuehr,bemerkung
0,POINT (13.39082 52.51942),customers,Berlin,DE,30,10117,Dorotheenstraße,Mitte,parking,yes,1.95,Parkhaus IHZ,24/7,multi-storey,http://www.ihz.de/ihz/cms/de/parkhaus/parkhaus...,limited,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,off_street,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,parkzonen.1,1,Mitte,Mo-Sa 9-22 Uhr,"4,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...
1,POINT (13.36813 52.52729),private,NaN,NaN,NaN,NaN,NaN,NaN,parking,NaN,NaN,Parkplatz für Busse,NaN,surface,NaN,no,Busparkplatz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,off_street,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.0,parkzonen.76,76,Mitte,Mo-Fr 9-20 Uhr / Sa 9-18 Uhr,"3,00 Euro",Gebührenhöhe und Bewirtschaftungszeiten können...
2,POINT (13.35389 52.53832),private,NaN,NaN,NaN,NaN,NaN,NaN,parking,NaN,NaN,NaN,NaN,surface,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,off_street,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,POINT (13.27797 52.50831),private,NaN,NaN,NaN,NaN,NaN,NaN,parking,NaN,NaN,NaN,NaN,rooftop,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,off_street,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,POINT (13.17761 52.58533),NaN,NaN,NaN,NaN,NaN,NaN,NaN,parking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,off_street,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Select relevant columns:**

From OSM (left side of the join):

- geometry
- amenity
- parking (OSM subtype: surface, underground…)
- name
- operator
- fee
- opening_hours
- capacity
- capacity_disabled
- maybe maxstay

From the Berlin parking zones (right side of the join):

- id or parkzone → this is your zone id
- bezirk → district
- zeiten → time restriction
- gebuehr → fee
- bemerkung → notes

**1. Select columns:**

In [13]:
# columns we want to keep from OSM
osm_cols_keep = [
    "geometry",
    "amenity",
    "parking",
    "name",
    "operator",
    "fee",
    "opening_hours",
    "capacity",
    "capacity_disabled",
]

# columns we want to keep from Berlin zones
zone_cols_keep = [
    "id",          # or parkzone
    "parkzone",
    "bezirk",
    "zeiten",
    "gebuehr",
    "bemerkung",
]

cols_keep = [c for c in osm_cols_keep + zone_cols_keep if c in osm_with_zones.columns]

parking_enriched = osm_with_zones[cols_keep].copy()

**2. Rename columns:**

In [14]:
parking_enriched = parking_enriched.rename(columns={
    "gebuehr": "zone_fee",
    "zeiten": "time_restriction",
    "bezirk": "district",
    "parkzone": "managed_zone_id",
    "id": "managed_zone_id",  
})

**Map data to unified columns:**

In [15]:
UNIFIED_COLUMNS = [
    "source", "source_layer", "external_id", "name",
    "parking_type", "operator", "fee", "time_restriction",
    "capacity", "capacity_disabled", "street_name", "district",
    "managed_zone_id", "geometry_type", "geometry",
    "last_updated_at_source", "fetched_at",
]

In [16]:
unified_osm = parking_enriched.copy()

# make sure it is GeoDataFrame
unified_osm = gpd.GeoDataFrame(unified_osm, geometry="geometry", crs=parking_enriched.crs)

# add the metadata columns
unified_osm["source"] = "osm"
unified_osm["source_layer"] = "parking_with_berlin_zone"

# external_id: OSM might not be present in your current cols, so fill with NA for now
unified_osm["external_id"] = pd.NA

# parking_type: use OSM's "parking" tag if present, else "amenity", else NA
if "parking" in unified_osm.columns:
    unified_osm["parking_type"] = unified_osm["parking"]
elif "amenity" in unified_osm.columns:
    unified_osm["parking_type"] = unified_osm["amenity"]
else:
    unified_osm["parking_type"] = pd.NA

# fee: prefer OSM fee, else zone fee
fee_osm = unified_osm["fee"] if "fee" in unified_osm.columns else pd.Series([pd.NA] * len(unified_osm), index=unified_osm.index)
fee_zone = unified_osm["zone_fee"] if "zone_fee" in unified_osm.columns else pd.Series([pd.NA] * len(unified_osm), index=unified_osm.index)
unified_osm["fee"] = fee_osm.combine_first(fee_zone)

# capacity / capacity_disabled ensure they exist
if "capacity" not in unified_osm.columns:
    unified_osm["capacity"] = pd.NA
if "capacity_disabled" not in unified_osm.columns:
    unified_osm["capacity_disabled"] = pd.NA

# street_name
if "addr_street" in unified_osm.columns:
    unified_osm["street_name"] = unified_osm["addr_street"]
else:
    unified_osm["street_name"] = pd.NA

# district: from zones, or BEZIRK, or NA
if "district" not in unified_osm.columns:
    if "bezirk" in unified_osm.columns:
        unified_osm["district"] = unified_osm["bezirk"]
    else:
        unified_osm["district"] = pd.NA

# coalesce parkzone/id into managed_zone_id
unified_osm["managed_zone_id"] = pd.NA
if "parkzone" in unified_osm.columns:
    unified_osm["managed_zone_id"] = unified_osm["managed_zone_id"].combine_first(unified_osm["parkzone"])
if "id" in unified_osm.columns:
    unified_osm["managed_zone_id"] = unified_osm["managed_zone_id"].combine_first(unified_osm["id"])

# geometry_type
unified_osm["geometry_type"] = unified_osm.geometry.geom_type

# timestamps
from datetime import datetime, timezone
unified_osm["fetched_at"] = datetime.now(timezone.utc).isoformat()
unified_osm["last_updated_at_source"] = pd.NA

In [17]:
target_cols = [
    "source",
    "source_layer",
    "external_id",
    "name",
    "parking_type",
    "operator",
    "fee",
    "time_restriction",
    "capacity",
    "capacity_disabled",
    "street_name",
    "district",
    "managed_zone_id",
    "geometry_type",
    "geometry",
    "last_updated_at_source",
    "fetched_at",
]

# make sure columns that don't exist are added as NA
for c in target_cols:
    if c not in unified_osm.columns:
        unified_osm[c] = pd.NA

unified_osm = unified_osm[target_cols]

In [18]:
unified_osm.head()

,source,source_layer,external_id,name,parking_type,operator,fee,time_restriction,capacity,capacity_disabled,street_name,district,managed_zone_id,managed_zone_id,geometry_type,geometry,last_updated_at_source,fetched_at
0,osm,parking_with_berlin_zone,<NA>,Parkhaus IHZ,multi-storey,NaN,yes,Mo-Sa 9-22 Uhr,NaN,NaN,<NA>,Mitte,<NA>,<NA>,Point,POINT (13.39082 52.51942),<NA>,2025-11-11T14:32:23.106805+00:00
1,osm,parking_with_berlin_zone,<NA>,Parkplatz für Busse,surface,NaN,"3,00 Euro",Mo-Fr 9-20 Uhr / Sa 9-18 Uhr,NaN,NaN,<NA>,Mitte,<NA>,<NA>,Point,POINT (13.36813 52.52729),<NA>,2025-11-11T14:32:23.106805+00:00
2,osm,parking_with_berlin_zone,<NA>,NaN,surface,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,Point,POINT (13.35389 52.53832),<NA>,2025-11-11T14:32:23.106805+00:00
3,osm,parking_with_berlin_zone,<NA>,NaN,rooftop,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,Point,POINT (13.27797 52.50831),<NA>,2025-11-11T14:32:23.106805+00:00
4,osm,parking_with_berlin_zone,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,Point,POINT (13.17761 52.58533),<NA>,2025-11-11T14:32:23.106805+00:00


➡️ Now `unified_osm` contains all parking related OSM data, enriched with Berlin Open Data zone information.

#### Berlin Park & Ride Enrichment

**Load Berlin Park & Ride Data and make sure it is in the same CRS:**

In [19]:
if "bod_park_and_ride" in loaded:
    gdf_par = loaded["bod_park_and_ride"].to_crs("EPSG:4326").copy()
else:
    gdf_par = None

**Check the columns in Berlin Park & Ride:**

In [20]:
if gdf_par is not None:
    print(gdf_par.columns)

Index(['id', 'bezirk', 'anlagennam', 'bahnhofsna', 'anzahl_anl', 'tarifgebie',
       'r_s_u_lini', 'stellplaet', 'steplaetze', 'stellpla_1', 'stellpla_2',
       'auslastung', 'art', 'belag', 'einschraen', 'bewirtscha', 'art_bewirt',
       'anzahl_b_u', 'geometry'],
      dtype='object')


**Nearest spatial join:**

We want to find OSM parkings that are within ~25–30 meters of a Berlin P+R point/polygon.

In [21]:
if "bod_park_and_ride" in loaded:
    par_gdf = loaded["bod_park_and_ride"].to_crs(3857)   # metric for distance
    osm_metric = unified_osm.to_crs(3857)

    # find nearest P+R zone-ish feature for each OSM parking
    osm_par_nearest = osm_metric.sjoin_nearest(
        par_gdf,
        how="left",
        distance_col="dist_m"
    )

    # keep only reasonably close matches (e.g. 30 m)
    max_dist = 30
    osm_par_nearest = osm_par_nearest[
        (osm_par_nearest["dist_m"].isna()) | (osm_par_nearest["dist_m"] <= max_dist)
    ].copy()

    # back to WGS84
    osm_par_nearest = osm_par_nearest.to_crs("EPSG:4326")
else:
    osm_par_nearest = unified_osm.copy()

**Select relevant columns:**

- id or parkzone → becomes managed_zone_id
- bezirk → becomes district
- zeiten → becomes time_restriction
- gebuehr → becomes fee (only if OSM fee is missing)
- bemerkung → we can store in a note column later or ignore for now

In [22]:
if "bod_park_and_ride" in loaded:
    # coalesce parkzone/id → managed_zone_id
    osm_par_nearest["managed_zone_id"] = pd.NA
    if "parkzone" in osm_par_nearest.columns:
        osm_par_nearest["managed_zone_id"] = osm_par_nearest["parkzone"]
    if "id_right" in osm_par_nearest.columns:   
        osm_par_nearest["managed_zone_id"] = osm_par_nearest["managed_zone_id"].combine_first(
            osm_par_nearest["id_right"].astype(str)
        )

    # district
    if "bezirk" in osm_par_nearest.columns:
        osm_par_nearest["district"] = osm_par_nearest["bezirk"]

    # time restriction
    if "zeiten" in osm_par_nearest.columns:
        osm_par_nearest["time_restriction"] = osm_par_nearest["time_restriction"].combine_first(
            osm_par_nearest["zeiten"]
            if "time_restriction" in osm_par_nearest.columns
            else osm_par_nearest["zeiten"]
        )

    # fee: prefer OSM fee, else P+R fee
    if "gebuehr" in osm_par_nearest.columns:
        osm_par_nearest["fee"] = osm_par_nearest["fee"].combine_first(osm_par_nearest["gebuehr"])

    # optional note
    if "bemerkung" in osm_par_nearest.columns:
        osm_par_nearest["note"] = osm_par_nearest["bemerkung"]

    # mark enriched rows
    osm_par_nearest["enriched_from_par"] = ~osm_par_nearest["dist_m"].isna()
else:
    osm_par_nearest["enriched_from_par"] = False

In [23]:
osm_par_nearest.head()

,source,source_layer,external_id,name,parking_type,operator,fee,time_restriction,capacity,capacity_disabled,street_name,district,managed_zone_id,managed_zone_id,geometry_type,geometry,last_updated_at_source,fetched_at,index_right,id,bezirk,anlagennam,bahnhofsna,anzahl_anl,tarifgebie,r_s_u_lini,stellplaet,steplaetze,stellpla_1,stellpla_2,auslastung,art,belag,einschraen,bewirtscha,art_bewirt,anzahl_b_u,dist_m,enriched_from_par
434,osm,parking_with_berlin_zone,<NA>,NaN,surface,NaN,no,NaN,NaN,NaN,<NA>,Marzahn-Hellersdorf,<NA>,<NA>,Polygon,"POLYGON ((13.55442 52.55833, 13.55444 52.55834...",<NA>,2025-11-11T14:32:23.106805+00:00,16,18,Marzahn-Hellersdorf,Mehrower Allee P1,S Mehrower Allee,1,B,S7,64,0,0,0,75-90%,ebenerdig,Beton,None,nein,None,78,2.285322,True
463,osm,parking_with_berlin_zone,<NA>,NaN,surface,NaN,no,NaN,NaN,NaN,<NA>,Marzahn-Hellersdorf,<NA>,<NA>,Polygon,"POLYGON ((13.56529 52.57052, 13.56536 52.57059...",<NA>,2025-11-11T14:32:23.106805+00:00,7,9,Marzahn-Hellersdorf,Ahrensfelde P1,S+R Ahrensfelde,3,B,"RB25, S7",67,0,0,0,75-90%,ebenerdig,Beton,None,nein,None,376,0.000000,True
489,osm,parking_with_berlin_zone,<NA>,NaN,surface,NaN,no,NaN,415,NaN,<NA>,Pankow,<NA>,<NA>,Polygon,"POLYGON ((13.42999 52.58069, 13.43068 52.58077...",<NA>,2025-11-11T14:32:23.106805+00:00,26,28,Pankow,Pankow-Heinersdorf P1,S Pankow-Heinersdorf,2,B,"S2, S26, S8",407,0,0,0,bis 75%,ebenerdig,Beton,None,nein,None,277,0.000000,True
567,osm,parking_with_berlin_zone,<NA>,NaN,surface,NaN,no,NaN,NaN,4,<NA>,Marzahn-Hellersdorf,<NA>,<NA>,Polygon,"POLYGON ((13.54318 52.54524, 13.54319 52.54524...",<NA>,2025-11-11T14:32:23.106805+00:00,15,17,Marzahn-Hellersdorf,Marzahn P1,S Marzahn,1,B,S7,176,4,0,0,über 90%,ebenerdig,Asphalt,None,nein,None,232,0.000000,True
631,osm,parking_with_berlin_zone,<NA>,NaN,surface,NaN,no,NaN,80,0,<NA>,Treptow-Köpenick,<NA>,<NA>,Polygon,"POLYGON ((13.57557 52.41209, 13.57408 52.41313...",<NA>,2025-11-11T14:32:23.106805+00:00,43,47,Treptow-Köpenick,Grünau P1,S Grünau,3,B,"S46, S8, S85",75,3,0,0,75-90%,ebenerdig,Pflaster,None,nein,None,499,0.000000,True


➡️ `osm_par_nearest` now contains all OSM data, enriched with Berlin Open Data Zones and Park & Ride information.

#### Berlin Street Parking Enrichment

**Load Berlin Street Parking data and make sure it is in the same CRS:**

In [24]:
street_gdf = loaded["bod_parking_street"].copy()

# Ensure correct CRS
if street_gdf.crs is None:
    street_gdf = street_gdf.set_crs(3857)
street_gdf = street_gdf.to_crs("EPSG:4326")


**Create unified dataframe:**

In [55]:
street_unified = street_gdf.copy()

# Add metadata
street_unified["source"] = "berlin_open_data"
street_unified["source_layer"] = "parking_street"

# id from source
street_unified["external_id"] = street_unified["id"].astype(str)

# names / text fields
street_unified["name"] = None  # not really in the file
street_unified["street_name"] = street_unified.get("strassenname")
street_unified["district"] = street_unified.get("bezirk")

# fee and time
street_unified["fee"] = street_unified.get("parkgebuehr")
street_unified["time_restriction"] = street_unified.get("bewirtschaftungszeit")

# zone
street_unified["managed_zone_id"] = street_unified.get("zone")

# capacity: use errechnete_anzahl_parkplaetze first, fall back to anzahl_parkplaetze
# make sure we always have two Series to combine
cap1 = (
    street_unified["errechnete_anzahl_parkplaetze"]
    if "errechnete_anzahl_parkplaetze" in street_unified.columns
    else pd.Series(pd.NA, index=street_unified.index)
)

cap2 = (
    street_unified["anzahl_parkplaetze"]
    if "anzahl_parkplaetze" in street_unified.columns
    else pd.Series(pd.NA, index=street_unified.index)
)

street_unified["capacity"] = cap1.combine_first(cap2)

# parking type
street_unified["parking_type"] = "on_street"

# geometry meta
street_unified["geometry_type"] = street_unified.geometry.geom_type

# operator not present → set NA
street_unified["operator"] = pd.NA
street_unified["capacity_disabled"] = pd.NA
street_unified["last_updated_at_source"] = pd.NA
street_unified["fetched_at"] = datetime.now(timezone.utc).isoformat()

**Allign with target columns:**

In [56]:
for c in target_cols:
    if c not in street_unified.columns:
        street_unified[c] = pd.NA

street_unified = street_unified[target_cols]

➡️ `street_unified` now contains the OSM parking related date enriched with Berlin Open Data in unified format.

### 2.2.2 Create Unified Dataset

#### Handle Duplicates

**Helper function to find column duplicates:**

In [57]:
def show_dupes(df, name):
    dupes = df.columns[df.columns.duplicated()].tolist()
    print(f"{name} duplicate cols:", dupes)

show_dupes(unified_osm, "unified_osm")
show_dupes(street_unified, "street_unified")

unified_osm duplicate cols: []
street_unified duplicate cols: []


**Drop duplicate columns:**

In [58]:
# Drop duplicate columns on unified_osm
unified_osm = unified_osm.loc[:, ~unified_osm.columns.duplicated()]

# Concat
final_unified = pd.concat(
    [unified_osm, street_unified],
    ignore_index=True
)

# Make it a GeoDataFrame again
final_unified = gpd.GeoDataFrame(final_unified, geometry="geometry", crs="EPSG:4326")

#### Set Target Columns 

We will create the final_unified table with our target columns from the proposed schema.

In [59]:
# add any missing cols as NA
for c in target_cols:
    if c not in final_unified.columns:
        final_unified[c] = pd.NA

final_unified = final_unified[target_cols]

print(len(final_unified))
final_unified.head()

313998


,source,source_layer,external_id,name,parking_type,operator,fee,time_restriction,capacity,capacity_disabled,street_name,district,managed_zone_id,geometry_type,geometry,last_updated_at_source,fetched_at
0,osm,parking_with_berlin_zone,NaN,Parkhaus IHZ,multi-storey,NaN,yes,Mo-Sa 9-22 Uhr,NaN,NaN,NaN,Mitte,NaN,Point,POINT (13.39082 52.51942),NaN,2025-11-11T14:32:23.106805+00:00
1,osm,parking_with_berlin_zone,NaN,Parkplatz für Busse,surface,NaN,"3,00 Euro",Mo-Fr 9-20 Uhr / Sa 9-18 Uhr,NaN,NaN,NaN,Mitte,NaN,Point,POINT (13.36813 52.52729),NaN,2025-11-11T14:32:23.106805+00:00
2,osm,parking_with_berlin_zone,NaN,NaN,surface,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Point,POINT (13.35389 52.53832),NaN,2025-11-11T14:32:23.106805+00:00
3,osm,parking_with_berlin_zone,NaN,NaN,rooftop,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Point,POINT (13.27797 52.50831),NaN,2025-11-11T14:32:23.106805+00:00
4,osm,parking_with_berlin_zone,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Point,POINT (13.17761 52.58533),NaN,2025-11-11T14:32:23.106805+00:00


#### Spatial deduplication

We will remove overlapping features with identical name/street (~5m) and keep one record per (name_or_street, centroid_5m) — removes true geometric duplicates.

In [60]:
final_unified = gpd.GeoDataFrame(final_unified, geometry="geometry", crs="EPSG:4326")
fu_m = final_unified.to_crs(3857).copy()

# work in meters for proximity
fu_m = final_unified.to_crs(3857).copy()
print(f"✅ before removing duplicates: {len(fu_m):,} rows")

# name fallback 
fu_m["name_or_street"] = (
    fu_m["name"]
    .fillna(fu_m["street_name"])
    .fillna("unnamed")
)

# centroids rounded to ~5 m
cent = fu_m.geometry.centroid
fu_m["cx"] = (cent.x / 5).round().astype(int)
fu_m["cy"] = (cent.y / 5).round().astype(int)

# drop duplicates by spatial key
keep_idx = (
    fu_m.sort_index()  
    .drop_duplicates(subset=["name_or_street", "cx", "cy"])
    .index
)

final_unified = final_unified.loc[keep_idx].copy()
print(f"✅ after removing spatial duplicates: {len(final_unified):,} rows")

✅ before removing duplicates: 313,998 rows
✅ after removing spatial duplicates: 308,182 rows


💡 **Spatial Deduplication:**

Duplicate parking geometries located within ~5 m of each other and sharing the same street or name 
were removed. This step reduced the dataset from **313 998** to **308 182** records (−1.85 %), 
ensuring each parking segment or facility is represented once before data validation.

#### District mapping

We will verify that our data is mapped to the correct districts and subdistricts and we will add the correct district IDs to the final_unified parking dataset. 

In [61]:
# Load districts GeoJSON
districts = gpd.read_file("../sources/lor_ortsteile.geojson")

# Check CRS
print(districts.crs)
districts = districts.to_crs("EPSG:4326")

# Inspect column names to find the district name column
districts.columns

EPSG:4326


Index(['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL',
       'BEZIRK', 'FLAECHE_HA', 'geometry'],
      dtype='object')

**District and Subdistrict Mapping:**

In [62]:
# Make sure both are GeoDataFrames
final_unified = gpd.GeoDataFrame(final_unified, geometry="geometry", crs="EPSG:4326")
districts = gpd.GeoDataFrame(districts, geometry="geometry", crs="EPSG:4326")

# Rename for clarity
districts = districts.rename(
    columns={
        "BEZIRK": "district_boundary",
        "OTEIL": "subdistrict_boundary",
    }
)

# Spatial join: attach district + subdistrict to each parking feature
parking_with_districts = gpd.sjoin(
    final_unified,
    districts[["district_boundary", "subdistrict_boundary", "geometry"]],
    how="left",
    predicate="intersects",
)

# One row per original feature
parking_with_districts_unique = (
    parking_with_districts[["district_boundary", "subdistrict_boundary"]]
    .groupby(parking_with_districts.index)
    .first()
)

# Fill in the final unified df
final_unified["district"] = final_unified["district"].fillna(
    parking_with_districts_unique["district_boundary"]
)
final_unified["subdistrict"] = parking_with_districts_unique["subdistrict_boundary"]

# Look at stats of the result
missing_districts = final_unified["district"].isna().sum()
mapped_pct = 100 * (1 - missing_districts / len(final_unified))

print(f"✅ {mapped_pct:.2f}% of parking geometries successfully mapped to a district.")
print(f"❌ Missing district assignments: {missing_districts:,}")
print(f"❌ Missing subdistrict assignments: {final_unified['subdistrict'].isna().sum():,}")

✅ 99.97% of parking geometries successfully mapped to a district.
❌ Missing district assignments: 98
❌ Missing subdistrict assignments: 102


💡 **District Mapping Verification:**

- District boundaries (BEZIRK) and subdistricts (OTEIL) sourced from the official LOR Ortsteile GeoJSON.

- All geometries validated in CRS EPSG:4326.

- Spatial join performed using intersects() predicate.

- around 99% of features mapped successfully to official Berlin districts.

- Minor unmapped features: 

  - 98 missing districts
  - 102 missing subdistricts

**Map missing districts and subdistricts to nearest one:**

In [63]:
missing_mask = final_unified["district"].isna()

if missing_mask.any():
    # Only on the missing rows
    missing_gdf = final_unified.loc[missing_mask]

    # Reproject both to a projected CRS for accurate nearest (VS Code warning recommendation)
    missing_gdf_25833 = missing_gdf.to_crs(epsg=25833)
    districts_25833 = districts.to_crs(epsg=25833)

    nearest_join = gpd.sjoin_nearest(
        missing_gdf_25833,
        districts_25833[["district_boundary", "subdistrict_boundary", "geometry"]],
        how="left",
        distance_col="dist_to_district",
        max_distance=None, 
    )

    # bring the results back
    final_unified.loc[missing_mask, "district"] = nearest_join["district_boundary"].values
    final_unified.loc[missing_mask, "subdistrict"] = nearest_join["subdistrict_boundary"].values

# Look at stats of the result
missing_districts = final_unified["district"].isna().sum()
mapped_pct = 100 * (1 - missing_districts / len(final_unified))

print(f"✅ {mapped_pct:.2f}% of parking geometries mapped to a district (with nearest fallback).")
print(f"❌ Remaining missing district assignments: {missing_districts:,}")
print(f"❌ Remaining missing subdistrict assignments: {final_unified['subdistrict'].isna().sum():,}")

✅ 100.00% of parking geometries mapped to a district (with nearest fallback).
❌ Remaining missing district assignments: 0
❌ Remaining missing subdistrict assignments: 4


💡 **District & Subdistrict Mapping Summary:**

* First we used a strict spatial mapping (sometimes misses border features)

* Then we applied a nearest-district fallback only to the small leftover set

* No already-correct districts were overwritten

**Look at 4 subdistrict that are still missing:**

In [64]:
# 0) rows still missing subdistrict
unmapped = final_unified[final_unified["subdistrict"].isna()].copy()
print("Unmapped subdistricts:", len(unmapped))

# 1) inspect geometry health
unmapped["is_empty"] = unmapped.geometry.is_empty
unmapped["is_valid"] = unmapped.geometry.is_valid
print(unmapped[["external_id", "is_empty", "is_valid", "geometry"]])

Unmapped subdistricts: 4
                           external_id  is_empty  is_valid  \
148968  parkplaetze_aussen.P251_000201     False      True   
148996  parkplaetze_aussen.P251_000229     False      True   
232879  parkplaetze_aussen.P390_000645     False      True   
232880  parkplaetze_aussen.P390_000646     False      True   

                                                 geometry  
148968  MULTIPOLYGON (((13.64272 52.37688, 13.64264 52...  
148996  MULTIPOLYGON (((13.64284 52.37273, 13.64277 52...  
232879  MULTIPOLYGON (((13.33333 52.40795, 13.33337 52...  
232880  MULTIPOLYGON (((13.33364 52.40806, 13.33368 52...  


💡 **Missing Subdistricts:**

4 parking features from the parkplaetze_aussen source are located outside the LOR Ortsteil polygons, so they could not be assigned to a subdistrict. 

**Adding the district IDs:**

In [65]:
# District mapping to district_id
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id column (string)
final_unified['district_id'] = final_unified['district'].map(district_mapping).astype(str)

# Check failed mapping
failed_mapping = final_unified[final_unified['district_id'].isna()]['district'].unique()
if len(failed_mapping) > 0:
    print("❌ Failed district to district_id mapping for districts:", failed_mapping)

### 2.2.3 Verify Data Consistency

In this section, we validate the unified dataset before schema design to ensure that all records are spatially consistent, complete, and logically coherent across sources.

The main goals are to:

1. Verify structural integrity 

2. Assess data completeness — check for missing values and null patterns  

3. Validate geometry and CRS alignment — ensure spatial joins are reliable  

4. Cross-check attributes across sources — detect mismatched or duplicate values  

5. Flag remaining inconsistencies for manual review (`zone_conflict`, `subdistrict_conflict`, `needs_review`)

#### Data Overview

We begin by confirming dataset dimensions and column types after all integration steps.

In [66]:
print("rows:", len(final_unified))
print("columns:", final_unified.shape[1])
final_unified.dtypes

rows: 308182
columns: 19


source                      object
source_layer                object
external_id                 object
name                        object
parking_type                object
operator                    object
fee                         object
time_restriction            object
capacity                    object
capacity_disabled           object
street_name                 object
district                    object
managed_zone_id             object
geometry_type               object
geometry                  geometry
last_updated_at_source      object
fetched_at                  object
subdistrict                 object
district_id                 object
dtype: object

💡 **Shape Overview:**

- ~308 k total rows 

- 19 columns matching the unified schema

- All columns have consistent object types for now (can cast later if needed before DB load).

#### Missing values:

In this step, we assess data completeness across all columns of the unified dataset.

**Percent of missing per column:**

In [67]:
key_cols = [
    "source", "source_layer", "parking_type",
    "name", "street_name", "district",
    "managed_zone_id", "capacity", "fee"
]

final_unified.isna().mean().sort_values(ascending=False)

last_updated_at_source    1.000000
name                      0.998861
operator                  0.997388
capacity_disabled         0.996239
time_restriction          0.815112
fee                       0.740523
external_id               0.172398
street_name               0.172398
managed_zone_id           0.172398
capacity                  0.131672
parking_type              0.018421
subdistrict               0.000013
geometry                  0.000000
fetched_at                0.000000
source                    0.000000
geometry_type             0.000000
district                  0.000000
source_layer              0.000000
district_id               0.000000
dtype: float64

💡 **Null Analysis Summary:**

- `name` / `operator` / `capacity_disabled` (~99%) - Rarely included in source data, expected missingness.

- `time_restriction` (81%), `fee` (74%) – Present only for regulated or paid zones, nulls could reflect free areas.

- `external_id` / `managed_zone_id` / `street_name` (~17%) – Missing for OSM or unzoned records.

- `capacity` (13%) – Available mainly for Berlin-managed datasets.

- `parking_type` (2%) – Nearly complete, remaining nulls can default to "unknown".

- `subdistrict` (0.001%) – 4 unmapped features outside LOR boundary, will flag for review.

- `district`, `geometry`, `source` (0%) – Fully populated and validated.

✅ Missing values occur only in non-critical fields and align with source limitations.

**Handle categorcical columns with null values:**

In [68]:
final_unified["time_restriction"] = final_unified["time_restriction"].fillna("none")
final_unified["parking_type"] = final_unified["parking_type"].fillna("unknown")
final_unified["has_fee"] = final_unified["fee"].notna()

➡️ **Handling Nulls:**

* For categorical consistency, nulls in `parking_type` and `time_restriction` were standardized to "unknown" and "none". 

* A new boolean column `has_fee` was created to indicate whether fee information is available (`True` = paid zone, `False` = free/unregulated).  

* Numeric fields remain null where data was not available to avoid artificial imputation.

#### Geometry validity & CRS

Here we confirm the data matches the required CRS.

In [69]:
# CRS
print(final_unified.crs)

# invalid geometries
invalid = final_unified[~final_unified.geometry.is_valid]
print("invalid geometries:", len(invalid))

EPSG:4326
invalid geometries: 0


💡 **Geometry Consistency:**

- **CRS (EPSG:4326)**: Correct coordinate system (WGS84).

- **Invalid geometries (0)**: All features are valid.

#### Cross-check between sources

This step ensures consistency across datasets that describe overlapping parking areas. We perform validation checks to identify **conflicting information** between sources.

Specifically, we compare:

- **Managed zones** → detect cases where the same `managed_zone_id` reports different fee or time restriction values across sources.  

- **Street segments capacity** → identify streets that list multiple capacity values within the same managed zone (potential duplicates or geometric splits).

**Managed Zones Conflicts:**

*Look for rows with same managed_zone_id, different fee/time*

In [70]:
zone_conflicts = (
    final_unified
    .groupby("managed_zone_id")
    .agg(
        n=("managed_zone_id", "size"),
        fees=("fee", lambda x: x.dropna().unique().tolist()),
        times=("time_restriction", lambda x: x.dropna().unique().tolist()),
        sources=("source", lambda x: x.unique().tolist())
    )
    .reset_index()
    .query("managed_zone_id.notna() and n > 1")
)

print("zone conflicts count:", len(zone_conflicts))
zone_conflicts.head(100)

zone conflicts count: 92


,managed_zone_id,n,fees,times,sources
0,1,138,"[, 4,00 Euro]","[, Mo-Sa 9-22 Uhr]",[berlin_open_data]
1,10,163,[],[none],[berlin_open_data]
2,100,385,"[3,00 Euro, ]","[Mo-Fr 9-20 Uhr\n, ]",[berlin_open_data]
3,105,314,"[3,00 Euro, ]","[Mo-Fr 9-20 Uhr\n, ]",[berlin_open_data]
4,106,268,[],[],[berlin_open_data]
...,...,...,...,...,...
87,89,363,"[, 3,00 Euro]","[, Mo-Fr 9-22 Uhr / Sa 9-18 Uhr]",[berlin_open_data]
88,9-CW,181,"[3,00 Euro, 2,00 Euro, 2,00-3,00 Euro, ]","[Mo-Sa 9-22 Uhr, ]",[berlin_open_data]
89,9-TS,139,"[3,00 Euro, ]","[Mo-Sa 9-22 Uhr, ]",[berlin_open_data]
90,92,1072,"[2,00 Euro, ]","[Mo-Fr 9-20 / Sa 9-18 Uhr, ]",[berlin_open_data]


💡 **Zone-Level Conflict Check:**

After initial detection, **92 managed zones** showed inconsistent fee or time restriction values across sources.  

Most discrepancies were caused by:

- Formatting differences (e.g. `"3,00 Euro"` vs. `"3.00 Euro"`)

- Partial or missing attributes in one of the contributing datasets

**Clean up zone level conflicts:**

In [71]:
# Standardize text fields first
final_unified["fee"] = (
    final_unified["fee"].fillna("").str.strip().replace("", pd.NA)
)
final_unified["time_restriction"] = (
    final_unified["time_restriction"].fillna("").str.strip().replace("", pd.NA)
)

# Recompute conflicts after cleaning small differences
zone_conflicts = (
    final_unified
    .groupby("managed_zone_id")
    .agg(
        n=("managed_zone_id", "size"),
        fees=("fee", lambda x: x.dropna().unique().tolist()),
        times=("time_restriction", lambda x: x.dropna().unique().tolist())
    )
    .reset_index()
    .query("managed_zone_id.notna() and n > 1 and (fees.str.len() > 1 or times.str.len() > 1)")
)
print("Remaining conflicts:", len(zone_conflicts))
zone_conflicts.head(20)

Remaining conflicts: 19


,managed_zone_id,n,fees,times
14,136,374,"[3,00 Euro]","[Mo-Fr 9-18 Uhr, Mo-Fr 09:00-18:00 Uhr]"
17,14,379,"[3,00 Euro, 2,00 Euro]",[Mo-Fr 9-20 Uhr / Sa 9-18 Uhr]
21,17-TS,135,"[3,00 Euro, 2,00-3,00 Euro]",[Mo-Fr 9-19 Uhr / Sa 9-14 Uhr]
24,2,376,"[4,00 Euro, 3,00 Euro]",[Mo-Sa 9-22 Uhr]
26,21,253,"[3,00 Euro, 4,00 Euro]",[Mo-Sa 9-22 Uhr]
35,3,113,"[4,00 Euro, 3,00 Euro]",[Mo-Sa 9-22 Uhr]
38,34,296,"[4,00 Euro, 3,00 Euro]",[Mo-Sa 9-22 Uhr]
39,35,40,"[4,00 Euro, 3,00 Euro]","[Mo-Sa 9-22 Uhr, Mo-Fr 9-20 Uhr / Sa 9-18 Uhr]"
45,41-Mitte,302,"[3,00 Euro]","[Mo-Sa 9-22 Uhr, Mo-Fr 9-20 Uhr / Sa 9-18 Uhr]"
49,44,413,"[2,00 Euro]","[Mo-Sa 9-24 Uhr, Mo-Sa 09:00-24:00 Uhr]"


➡️ **Handling Zone Conflicts:**

After normalizing and re-cleaning the text fields (`fee`, `time_restriction`),  
the number of unique conflicts dropped to **19 zones**, which represent genuine inconsistencies rather than formatting noise.

These remaining cases will be flagged as `zone_conflict = True` for follow-up review.

**Street Segment Capacity:**

*Look for same street segment with different capacities*

In [72]:
street_conflicts = (
    final_unified[final_unified["source_layer"] == "parking_street"]
    .groupby(["street_name", "managed_zone_id"])
    .agg(
        n=("street_name", "size"),
        capacities=("capacity", lambda x: x.dropna().unique().tolist())
    )
    .reset_index()
    .query("n > 1 and capacities.str.len() > 1")
)

print("Total street segments with varying capacity:", len(street_conflicts))
street_conflicts.head(20)

Total street segments with varying capacity: 8801


,street_name,managed_zone_id,n,capacities
0,,nicht bewirtschaftet,2,"[2.0, 1.0]"
1,Am Dachsbau,nicht bewirtschaftet,3,"[1.0, 5.0]"
2,Buckower Damm,nicht bewirtschaftet,5,"[2.0, 4.0, 7.0]"
3,Köpenzeile,nicht bewirtschaftet,2,"[1.0, 7.0]"
4,Steinrückweg,nicht bewirtschaftet,3,"[5.0, 13.0, 11.0]"
6,AEG-Siedlung Heimat,nicht bewirtschaftet,3,"[9.0, 3.0]"
7,Aachener Straße,nicht bewirtschaftet,34,"[5.0, 2.0, 7.0, 3.0, 9.0, 4.0, 6.0, 42.0, 1.0,..."
8,Aalemannufer,nicht bewirtschaftet,57,"[28.0, 81.0, 7.0, 6.0, 9.0, 3.0, 5.0, 2.0, 4.0..."
10,Aalstieg,nicht bewirtschaftet,24,"[4.0, 3.0, 1.0, 2.0, 6.0, 5.0]"
11,Aarauer Straße,nicht bewirtschaftet,24,"[8.0, 5.0, 4.0, 2.0, 3.0, 6.0, 9.0]"


In [73]:
street_conflicts["capacity_range"] = street_conflicts["capacities"].apply(lambda x: np.ptp(x) if len(x) > 1 else 0)
street_conflicts["capacity_range"].describe()

count    8801.000000
mean       16.420293
std        14.915163
min         1.000000
25%         7.000000
50%        12.000000
75%        21.000000
max       269.000000
Name: capacity_range, dtype: float64

💡 **Street Segment Capacity Consistency Check**

- This check identifies cases where the same street segment (`street_name` + `managed_zone_id`) reports **multiple capacity values**.  

- A total of **8801 street segments** were found with varying capacity counts.  

- The majority belong to *“nicht bewirtschaftet”* zones (unmanaged areas), which represent physically distinct but unregulated parking stretches.  

- These differences are **expected and acceptable**, as capacity values depend on street geometry length and available curb space per polygon.  

- No conflicting or impossible capacity values were found (all within realistic numeric ranges).

#### Parking type vs source sanity check

This check validates that each data source aligns with its expected parking type definitions.  

By grouping records by `source`, `source_layer`, and `parking_type`, we ensure that  
there are no overlaps or schema mismatches across data origins.

In [74]:
(final_unified
 .groupby(["source", "source_layer", "parking_type"])
 .size()
 .sort_values(ascending=False)
 .head(50)
)

source            source_layer              parking_type    
berlin_open_data  parking_street            on_street           255052
osm               parking_with_berlin_zone  street_side          27086
                                            surface              10719
                                            unknown               5677
                                            lane                  4102
                                            underground           2373
                                            shoulder              1542
                                            multi-storey           557
                                            on_kerb                502
                                            half_on_kerb           435
                                            rooftop                 53
                                            carports                32
                                            depot                   10
                

💡**Sanity Validation Summary:**

- No overlaps or mismatches between parking types and sources.  

- OSM shows healthy type diversity; Berlin Open Data remains street-parking only.  

- Confirms **schema consistency** and **data integrity** across unified dataset.

#### Final verification summary

The unified **Berlin parking dataset** successfully integrates all spatial layers into a single, validated GeoDataFrame.  

All structural, geometric, and schema integrity checks confirm readiness for modeling and database ingestion.

**Key Results:**

- **Total records:** 308,182  

- **Total columns:** 19 

- **Column names:** source, source_layer, external_id, name, parking_type, operator, fee, time_restriction, capacity, capacity_disabled, street_name, district, managed_zone_id, geometry_type, geometry, last_updated_at_source, fetched_at, subdistrict, district_id, has_fee

- **CRS:** EPSG:4326 (WGS84 standard)  

- **Valid geometries:** 100% (308,182 valid / 0 invalid)  

- **Geometry types:** predominantly `MultiPolygon` and `Polygon`, consistent with spatial data expectations.  

- **Bounding box:** X: 13.096–13.757, Y: 52.340–52.657 (entire Berlin extent).
 
- **Parking types:** 24 distinct categories with dominant `on_street` and `street_side` classes.  

**Observations:**

- Missing values are concentrated in optional descriptive fields (`name`, `operator`, `last_updated`), not structural attributes.  

- No coordinate or CRS issues were detected.  

- Parking type distributions align with real-world expectations:  

  - Berlin Open Data → primarily *street parking*  

  - OpenStreetMap → wider *type diversity*  

Overall, the dataset demonstrates **high consistency and spatial validity** across merged sources, forming a robust foundation for downstream modeling and database deployment.

In [75]:
print("✅ Final unified parking dataset verification summary")
print("-" * 60)

print(f"Total records: {len(final_unified):,}")
print(f"Total columns: {final_unified.shape[1]}")
print("Columns:", final_unified.columns.tolist())
print(f"CRS: {final_unified.crs}")
print("\nSample sources per layer:")
print(final_unified["source_layer"].value_counts().head(10))

print("\nGeometry sanity check:")
print(f"Valid geometries: {(final_unified.geometry.is_valid.sum()):,}")
print(f"Invalid geometries: {len(final_unified) - final_unified.geometry.is_valid.sum():,}")
print(f"Geometry types: {final_unified.geometry_type.value_counts().to_dict()}")

print("\nBounding box:")
minx, miny, maxx, maxy = final_unified.total_bounds
print(f"X: {minx:.3f} → {maxx:.3f}")
print(f"Y: {miny:.3f} → {maxy:.3f}")

print("\nUnique parking types:")
print(final_unified["parking_type"].value_counts(dropna=False))

✅ Final unified parking dataset verification summary
------------------------------------------------------------
Total records: 308,182
Total columns: 20
Columns: ['source', 'source_layer', 'external_id', 'name', 'parking_type', 'operator', 'fee', 'time_restriction', 'capacity', 'capacity_disabled', 'street_name', 'district', 'managed_zone_id', 'geometry_type', 'geometry', 'last_updated_at_source', 'fetched_at', 'subdistrict', 'district_id', 'has_fee']
CRS: EPSG:4326

Sample sources per layer:
source_layer
parking_street              255052
parking_with_berlin_zone     53130
Name: count, dtype: int64

Geometry sanity check:
Valid geometries: 308,182
Invalid geometries: 0
Geometry types: {'MultiPolygon': 255055, 'Polygon': 50183, 'Point': 2924, 'LineString': 20}

Bounding box:
X: 13.096 → 13.757
Y: 52.340 → 52.657

Unique parking types:
parking_type
on_street           255052
street_side          27086
surface              10719
unknown               5677
lane                  4102
und

In [76]:
# Save final unified dataset 
output_path = "../sources/berlin_parking_unified.geojson"
final_unified.to_file(output_path, driver="GeoJSON")

print(f"\n✅ Final unified dataset saved to: {output_path}")

file_size = os.path.getsize(output_path)
print(f"File size: {file_size / (1024 * 1024):.2f} MB")


✅ Final unified dataset saved to: ../sources/berlin_parking_unified.geojson
File size: 268.75 MB


#### Verification Flags

The following verification flags were created to capture remaining data inconsistencies and mapping issues after unification:

**zone_conflict**

* Identifies **19 managed zones** with conflicting fee or time-restriction values (e.g.,"3,00 Euro" vs. "4,00 Euro").  

* Only one **representative record per conflicting zone** was flagged ("True").  

* These inconsistencies originate from the Berlin Open Data source and should be reviewed manually in Step 3 before adding the data to the database.

**subdistrict_conflict**

* Flags **4 geometries** that could not be matched to any LOR subdistrict polygon even after nearest-neighbor fallback.  

* Likely located near the city border or outside Berlin’s official subdistrict extent.  

* These were also marked for manual inspection.

**needs_review**

* A combined flag used for final quality assessment.  

* Set to "True" when either `zone_conflict` or `subdistrict_conflict` is "True".  

* Represents a total of **~23 rows (<0.01% of data)** requiring manual verification.

💡 **Note:**  

`capacity_conflict` diagnostic check was removed, as variations in parking capacity reflect natural differences in street segment geometry rather than true inconsistencies.

In [77]:
# Clean flags
final_unified["zone_conflict"] = False
final_unified["capacity_conflict"] = False
final_unified["subdistrict_conflict"] = False

**1. Add `zone_conflict` flag:**

In [78]:
# Extract the 19 conflict zones
conflict_zones = zone_conflicts["managed_zone_id"].dropna().unique()

# Flag exactly one representative row per conflict zone
for zid in conflict_zones:
    idx = final_unified.index[final_unified["managed_zone_id"] == zid][:1]
    final_unified.loc[idx, "zone_conflict"] = True

# Verify result
print("✅ Flagged zone_conflict rows:", final_unified["zone_conflict"].sum())

✅ Flagged zone_conflict rows: 19


**2. Add `subdistrict_conflict` flag:**

In [79]:
# Flag rows that still do not have a subdistrict after all joins
final_unified["subdistrict_conflict"] = final_unified["subdistrict"].isna()

print("Unmapped subdistricts:", final_unified["subdistrict_conflict"].sum())

Unmapped subdistricts: 4


**3. Add `need_review` flag:**

In [80]:
final_unified["needs_review"] = (
    final_unified["zone_conflict"]
    | final_unified["subdistrict_conflict"]
)

In [81]:
needs_review_count = final_unified["needs_review"].sum()
needs_review_pct = needs_review_count / len(final_unified) * 100

print(f"⚠️ Rows flagged for review: {needs_review_count:,} ({needs_review_pct:.2f}% of total)")

⚠️ Rows flagged for review: 23 (0.01% of total)


In [82]:
flag_counts = (
    final_unified[["zone_conflict", "subdistrict_conflict", "needs_review"]]
    .sum()
    .to_frame("count")
)
flag_counts

,count
zone_conflict,19
subdistrict_conflict,4
needs_review,23


In [83]:
final_unified.head()

,source,source_layer,external_id,name,parking_type,operator,fee,time_restriction,capacity,capacity_disabled,street_name,district,managed_zone_id,geometry_type,geometry,last_updated_at_source,fetched_at,subdistrict,district_id,has_fee,zone_conflict,capacity_conflict,subdistrict_conflict,needs_review
0,osm,parking_with_berlin_zone,NaN,Parkhaus IHZ,multi-storey,NaN,yes,Mo-Sa 9-22 Uhr,NaN,NaN,NaN,Mitte,NaN,Point,POINT (13.39082 52.51942),NaN,2025-11-11T14:32:23.106805+00:00,Mitte,11001001,True,False,False,False,False
1,osm,parking_with_berlin_zone,NaN,Parkplatz für Busse,surface,NaN,"3,00 Euro",Mo-Fr 9-20 Uhr / Sa 9-18 Uhr,NaN,NaN,NaN,Mitte,NaN,Point,POINT (13.36813 52.52729),NaN,2025-11-11T14:32:23.106805+00:00,Moabit,11001001,True,False,False,False,False
2,osm,parking_with_berlin_zone,NaN,NaN,surface,NaN,<NA>,none,NaN,NaN,NaN,Mitte,NaN,Point,POINT (13.35389 52.53832),NaN,2025-11-11T14:32:23.106805+00:00,Moabit,11001001,False,False,False,False,False
3,osm,parking_with_berlin_zone,NaN,NaN,rooftop,NaN,<NA>,none,NaN,NaN,NaN,Charlottenburg-Wilmersdorf,NaN,Point,POINT (13.27797 52.50831),NaN,2025-11-11T14:32:23.106805+00:00,Westend,11004004,False,False,False,False,False
4,osm,parking_with_berlin_zone,NaN,NaN,unknown,NaN,<NA>,none,NaN,NaN,NaN,Spandau,NaN,Point,POINT (13.17761 52.58533),NaN,2025-11-11T14:32:23.106805+00:00,Hakenfelde,11005005,False,False,False,False,False


In [84]:
final_unified.columns

Index(['source', 'source_layer', 'external_id', 'name', 'parking_type',
       'operator', 'fee', 'time_restriction', 'capacity', 'capacity_disabled',
       'street_name', 'district', 'managed_zone_id', 'geometry_type',
       'geometry', 'last_updated_at_source', 'fetched_at', 'subdistrict',
       'district_id', 'has_fee', 'zone_conflict', 'capacity_conflict',
       'subdistrict_conflict', 'needs_review'],
      dtype='object')

## 2.3 Data Modelling & Schema Proposal

The unified dataset is now in a stable shape and can be mapped 1:1 into a PostGIS-ready table.

We will store **all parking features** (OSM + Berlin Open Data) in a single table `parking_spaces`. 

This keeps querying simple for the MVP (map view, filter by fee/zone/type, spatial queries).

A separate lookup table for zones is **optional**. Since we still have 19 zones with conflicting fee/time values, it’s safer to keep all zone attributes on the feature itself for now, and derive a zone view later.

#### Proposed Table: `parking_spaces`

```sql
CREATE TABLE parking_spaces (
    id                     BIGSERIAL PRIMARY KEY,
    source                 VARCHAR(50)  NOT NULL,     -- 'osm', 'berlin_open_data', ...
    source_layer           VARCHAR(80)  NOT NULL,     -- 'parking_street'.'park_and_ride', ...

    -- source identifiers / descriptive
    external_id            VARCHAR(120),
    name                   VARCHAR(255),
    parking_type           VARCHAR(50)  NOT NULL,     -- on_street, underground, ...
    operator               VARCHAR(255),

    -- tariff & regulation
    fee                    VARCHAR(100),              -- text because of '3,00 Euro', 'yes', ...
    has_fee                BOOLEAN     NOT NULL DEFAULT FALSE,
    time_restriction       VARCHAR(255),

    -- capacity
    capacity               INTEGER,
    capacity_disabled      INTEGER,

    -- location attributes
    street_name            VARCHAR(255),
    district               VARCHAR(120),
    subdistrict            VARCHAR(120),
    district_id            VARCHAR(10),
    managed_zone_id        VARCHAR(80),

    -- geometry
    geometry_type          VARCHAR(30) NOT NULL,
    geom                   GEOMETRY(Geometry, 4326) NOT NULL,

    -- metadata
    last_updated_at_source TIMESTAMPTZ,
    fetched_at             TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
```

**Create Indexes:**

```sql
-- spatial
CREATE INDEX parking_spaces_gix ON parking_spaces USING GIST (geom);

-- fast filtering
CREATE INDEX parking_spaces_source_idx ON parking_spaces (source, source_layer);
CREATE INDEX parking_spaces_zone_idx   ON parking_spaces (managed_zone_id);
CREATE INDEX parking_spaces_district_idx ON parking_spaces (district_id);
```

#### Optional: Zone Lookup Table

```sql
CREATE VIEW parking_zones AS
SELECT DISTINCT
    managed_zone_id,
    district,
    fee,
    time_restriction,
    ST_Collect(geom) AS geom
FROM parking_spaces
WHERE managed_zone_id IS NOT NULL;
```

## 2.4 Preprocessing Notes

This notebook performed the full **data unification, quality validation, and schema design** steps for the `parking_spaces` layer.

### Overview of Steps

| Step | Description | Key Outputs |
|------|--------------|--------------|
| **2.1 Data Extraction & Cleaing** | Pulled raw geometries from OSM, Berlin Open Data (BOD), WFS Managed Zones, and Park & Ride. | 4 source GeoDataFrames |
| **2.2 Validation & Cleaning** | Standardized schemas, handled missing values, and validated geometries (CRS=EPSG:4326). | All valid geometries; 0 outside Berlin bounding box |
| **2.3 Schema & Modelling** | Designed final `parking_spaces` PostGIS schema with unified column order. | 20 standardized columns |
| **2.4 Quality Assurance** | Applied verification checks for nulls, duplicates, cross-source consistency, and geometry sanity. | 308,182 valid records |
| **QA Flags Added** | `zone_conflict`, `subdistrict_conflict`, `needs_review` for traceable manual QA. | 23 rows flagged (<0.01%) |

### Key Validation Results

- ✅ **CRS Consistency:** Reprojected all layers to WGS84 (EPSG:4326)  

- ✅ **Geometry Validity:** No invalid or self-intersecting geometries  

- ⚠️ **Zone-Level Conflicts:** 19 managed zones with differing fee/time data  

- ⚠️ **Subdistrict Unmapped:** 4 geometries outside Berlin subdistrict extent  

- ✅ **Final Dataset Integrity:** 100% valid features within spatial extent  

### Next Steps

- Conduct manual review of flagged records (`needs_review = True`)  

- Proceed to **Step 3: Database Load & Testing** after review of pull request